# Analysis for Multimodal CAD Editing Benchmark

Data formatting and basic preprocessing are performed in `edit_v2_wrangler.ipynb`

In [ ]:
import os
import sys
module_path = os.path.abspath(os.path.join('../..'))
sys.path.append(module_path)


import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

import ast


### Plot aesthetics

In [ ]:
# Aesthetic seaborn styling - clean with sparse, faint horizontal gridlines
sns.set_theme(style="whitegrid", font_scale=1.1)
sns.set_style({
    "axes.grid": True,
    "axes.grid.axis": "y",  # Only horizontal gridlines
    "grid.linestyle": "-",
    "grid.alpha": 0.25,
    "grid.color": "#e0e0e0",
    "axes.spines.left": False,
    "axes.spines.bottom": True,
    "axes.spines.right": False,
    "axes.spines.top": False,
    "axes.edgecolor": "#333333",
    # "patch.edgecolor": "none"
})
plt.rcParams["axes.axisbelow"] = True  # Grid behind data

# ============================================================
# Color palettes and orders for categorical comparisons
# ============================================================

# Difficulty: intuitive traffic-light progression
DIFFICULTY_ORDER = ["easy", "medium", "hard"]
DIFFICULTY_PALETTE = {"easy": "#4CAF50", "medium": "#FFA726", "hard": "#EF5350"}  # green → orange → red

# Assembly: False=single part (lighter), True=assembly (darker/richer)
ASSEMBLY_ORDER = [False, True]
ASSEMBLY_PALETTE = {False: "#90CAF9", True: "#1565C0"}  # light blue → dark blue

# Parametric: False=direct modeling (lighter), True=parametric (darker/richer)
PARAMETRIC_ORDER = [False, True]
PARAMETRIC_PALETTE = {False: "#CE93D8", True: "#7B1FA2"}  # light purple → dark purple

# Modality: interactive, text, draw-temp, draw-static
MODALITY_ORDER = ["text", "interactive", "draw-temp", "draw-static"]
# orange -> black -> red -> blue
MODALITY_PALETTE = {"interactive": "#FFA500", "text": "#000000", "draw-temp": "#FF5555", "draw-static": "#FF0000"}

USER_ORDER = ["gt_human", "other_human", "claude-sonnet-4.5_cadquery-script", "gemini-3-pro_cadquery-script", "gpt-5.2_cadquery-script"]
USER_PALETTE = {
    "gt_human": "#a0a0a0",   # ligher grey for ground truth human
    "other_human": "#777777",   # grey for other human
    "gpt-5.2_cadquery-script": '#49b599', # "#149e7b",        # teal for gpt-5.2
    "gemini-3-pro_cadquery-script": "#4b85f6",   # blue for gemini
    "claude-sonnet-4.5_cadquery-script": "#D97757" # orange for claude
}

# Use short labels for the x-axis for better readability
short_user_labels = {
    "other_human": "human baseline",
    "gt_human": "requester",
    "gpt-5.2_cadquery-script": "GPT-5.2",
    "gemini-3-pro_cadquery-script": "Gemini-3-Pro",
    "claude-sonnet-4.5_cadquery-script": "Claude-Sonnet-4.5",
}

font_size = 16

### load data

In [ ]:
date = "2026_03_18"

df_requests = pd.read_csv("../../consolidated_data/edits_v2_df_requests_{}.csv".format(date))
df_edits = pd.read_csv("../../consolidated_data/edits_v2_df_edits_{}.csv".format(date))
df_ratings = pd.read_csv("../../consolidated_data/edits_v2_df_ratings_{}.csv".format(date))

### basic preprocessing and columns

In [ ]:
def safe_literal_eval(val):
    if pd.isna(val) or val == "nan":
        return None
    try:
        return ast.literal_eval(val)
    except Exception:
        return None

In [ ]:
#### requests
df_requests['transcript_object'] = df_requests.transcript.apply(safe_literal_eval)

# count the number of segments in the transcript_object (skipping None)
df_requests['num_segments'] = df_requests.transcript_object.apply(lambda x: len(x['segments']) if x is not None else None)
df_requests.num_segments.value_counts()

# # count the number of words in the transcript_object
df_requests['num_words_transcript'] = df_requests.transcript_object.apply(lambda x: sum([len(y['words']) for y in x['segments']]) if x is not None else None)

# count the number of words in the text column when modality is text
# df_requests['num_words_text'] = df_requests.query("modality == 'text'").text.apply(lambda x: len(x.split()))
df_requests['num_words_text'] = df_requests['text'].apply(lambda x: len(x.split()) if isinstance(x, str) else None)

# create a new column for the number of words in the text column when modality is text, and the number of words in the transcript_object otherwise
df_requests['num_words_combined'] = df_requests.apply(lambda row: row['num_words_text'] if row['modality'] == 'text' else row['num_words_transcript'], axis=1)


#### edits
df_edits["edit_duration"] = df_edits["end_time"] - df_edits["start_time"]
df_edits['events'] = df_edits.events.apply(safe_literal_eval)
df_edits['n_events'] = df_edits.events.apply(lambda x: len(x) if x is not None else None)


#### add edit info into requests df
# average n events
df_requests['avg_n_events'] = df_requests['_id'].map(df_edits[df_edits.is_human].groupby('request')['n_events'].mean())


#### add request info into edits df

# merge num_words_combined from df_requests to df_edits where request_id (df_edits) == _id (df_requests)
df_edits['num_words_combined'] = df_edits['request'].map(df_requests.set_index('_id')['num_words_combined'])

# merge request_video_duration from df_requests to df_edits where request_id (df_edits) == _id (df_requests)
df_edits['request_video_duration'] = df_edits['request'].map(df_requests.set_index('_id')['request_video_duration'])
df_edits['edit_duration_over_request_video_duration'] = df_edits['edit_duration'] / df_edits['request_video_duration']

In [ ]:
# helper columns and dataframes for analysis

# create a subset for analysis of human data
df_edits_human = df_edits[df_edits.is_human]
df_edits_model = df_edits[~df_edits.is_human]


# group all human users together

def get_model_users(row):
    user = row['user']
    gt_human = row.get('gt_human', None)
    if isinstance(user, str) and any(bot_name in user.lower() for bot_name in ['claude', 'gemini', 'gpt-']):
        return user
    elif gt_human is True:
        return 'gt_human'
    elif gt_human is False:
        return 'other_human'
    else:
        return None

df_edits['model_users'] = df_edits.apply(get_model_users, axis=1)

## Dataset statistics

In [ ]:
# print number of requests
print(len(df_requests), "requests")
# print number of human edits
print(len(df_edits_human), "human edits")
# print number of non-human edits
print(len(df_edits) - len(df_edits_human), "model edits")
print("\n")

# Create a 2x2 grid crosstab of assembly and parametric
crosstab = pd.crosstab(df_requests['assembly'], df_requests['parametric'])
print(crosstab)
print("\n")

df_requests.modality.value_counts()

## Comparing modalities in requests and edits

In [ ]:
# number of words in transcript/ text

g = sns.catplot(data=df_requests,
    hue='difficulty',
    hue_order=DIFFICULTY_ORDER,
    palette=DIFFICULTY_PALETTE,
    y='num_words_combined',
    x='modality',
    order=MODALITY_ORDER,
    kind='bar',
    errorbar=None,
    legend=False,
    alpha=0.85,
    aspect=1,
)


# Update x-tick labels: replace "-" with new lines
ax = g.axes[0, 0]
xticklabels = [label.get_text().replace("-", "\n") for label in ax.get_xticklabels()]
ax.set_xticklabels(xticklabels, fontsize=font_size)


plt.xlabel("", fontsize=font_size)
plt.ylabel("recorded events", fontsize=font_size)
plt.xticks(fontsize=font_size)
plt.yticks(fontsize=font_size)
plt.xlabel("")
plt.ylabel("words in request", fontsize=font_size)

# plt.savefig("../../../figures/num_words_by_modality_and_difficulty.pdf", bbox_inches='tight')

In [ ]:
# time taken to produce request
# number of words in transcript

g = sns.catplot(data=df_requests,
    hue='difficulty',
    hue_order=DIFFICULTY_ORDER,
    palette=DIFFICULTY_PALETTE,
    y='request_video_duration',
    x='modality',
    order=MODALITY_ORDER,
    kind='bar',
    errorbar=None,
    legend=False,
    alpha=0.85,
    aspect=1,
)

# Update x-tick labels: replace "-" with new lines
ax = g.axes[0, 0]
xticklabels = [label.get_text().replace("-", "\n") for label in ax.get_xticklabels()]
ax.set_xticklabels(xticklabels, fontsize=font_size)


plt.xlabel("", fontsize=font_size)
plt.ylabel("recorded events", fontsize=font_size)
plt.xticks(fontsize=font_size)
plt.yticks(fontsize=font_size)
plt.xlabel("")
plt.ylabel("request duration (seconds)", fontsize=font_size)




# plt.savefig("../../../figures/request_video_duration_by_modality_and_difficulty.pdf", bbox_inches='tight')

In [ ]:
# time taken to produce edit
plt.figure(figsize=(8, 8))

# Barplot
ax = sns.catplot(
    data=df_edits_human,
    kind='bar',
    x='request_modality',
    order=MODALITY_ORDER,
    y='edit_duration',
    hue='request_difficulty',
    hue_order=DIFFICULTY_ORDER,
    errorbar=None,
    palette=DIFFICULTY_PALETTE,
    # col = "gt_human",
    # col_order=[True, False],
    alpha=0.85,
    aspect=1,
    legend=False,
)

# Update x-tick labels: replace "-" with new lines
for axes in ax.axes.flat:
    xticklabels = [label.get_text().replace("-", "\n") for label in axes.get_xticklabels()]
    axes.set_xticklabels(xticklabels)

plt.xlabel("")
plt.ylabel("edit duration (seconds)", fontsize=font_size)
plt.xticks(fontsize=font_size)
plt.yticks(fontsize=font_size)


# Add horizontal lines at 120s, 240s, 600s in difficulty palette colors to each facet
for ax_ in ax.axes.flat:
    for yval, color in zip([120, 240, 600], [DIFFICULTY_PALETTE[d] for d in DIFFICULTY_ORDER]):
        ax_.axhline(y=yval, color=color, linestyle='--', linewidth=1.2, alpha=0.9)

# plt.savefig("../../../figures/edit_duration_by_modality_and_difficulty.pdf", bbox_inches='tight')

In [ ]:
# avg number of events following request

font_size = 16
g = sns.catplot(data=df_requests,
    hue='difficulty',
    hue_order=DIFFICULTY_ORDER,
    palette=DIFFICULTY_PALETTE,
    y='avg_n_events',
    x='modality',
    order=MODALITY_ORDER,
    errorbar=None,
    kind='bar',
    alpha=0.85,
    aspect=1,
    legend=False,
)

# Update x-tick labels: replace "-" with new lines
ax = g.axes[0, 0]
xticklabels = [label.get_text().replace("-", "\n") for label in ax.get_xticklabels()]
ax.set_xticklabels(xticklabels, fontsize=font_size)

plt.xlabel("")

plt.xlabel("", fontsize=font_size)
plt.ylabel("recorded events", fontsize=font_size)
plt.xticks(fontsize=font_size)
plt.yticks(fontsize=font_size)


plt.savefig("../../../figures/num_events_by_modality_and_difficulty.pdf", bbox_inches='tight')

# Model accuracy 

In [ ]:
# human acceptance by modality

# metric = "rating_avg_human_instruction"
# metric = "rating_avg_human_quality"
metric = "human_acceptance_post_avg"

# plt.figure(figsize=(4, 8))

# model type- are models better at parametric models?
g1 = sns.catplot(
    df_edits,
    x='model_users',
    order=USER_ORDER,
    hue='request_modality',
    hue_order=MODALITY_ORDER,
    palette=MODALITY_PALETTE,
    y=metric,
    kind='bar',
    aspect=1.8,
    alpha=0.85,
    errorbar=None
)


# g1.axes is a numpy array of Axes if there are multiple facets (e.g. col, row arguments used)
# Here, adapt to possibly multiple axes (e.g. for different modalities being faceted)

# This will flatten axes for 1D and 2D facetting (e.g. col or row, or both)
axes = g1.axes.flatten() if hasattr(g1, "axes") else [g1.ax]

for ax in axes:
    xticklabels = ax.get_xticklabels()
    new_labels = []
    for label in xticklabels:
        txt = label.get_text()
        new_labels.append(short_user_labels.get(txt, txt))
    ax.set_xticklabels(new_labels, fontsize=14)
    ax.set_xlabel("", fontsize=14)
    ax.tick_params(axis='y', labelsize=14)

plt.ylim(0, 1)
g1._legend.remove()
axes[0].legend(loc='upper right', title='request modality', fontsize=14, title_fontsize=1)
plt.ylabel("human acceptance", fontsize=14)

# # Update x-tick labels: replace "-" with new lines
# ax = g.axes[0, 0]
# xticklabels = [label.get_text().replace("-", "\n") for label in ax.get_xticklabels()]
# ax.set_xticklabels(xticklabels)


# plt.savefig("../../../figures/model_performance_by_modality_{}.pdf".format(metric), bbox_inches='tight')

In [ ]:
# Human acceptance by difficulty

font_size = 14

# metric = "rating_avg_human_instruction"
# metric = "rating_avg_human_quality"
metric = "human_acceptance_post_avg"

# plt.figure(figsize=(4, 8))

# model type- are models better at parametric models?
g1 = sns.catplot(
    df_edits,
    x='model_users',
    order=USER_ORDER,
    hue='request_difficulty',
    hue_order=DIFFICULTY_ORDER,
    palette=DIFFICULTY_PALETTE,
    y=metric,
    kind='bar',
    aspect=1.8,
    alpha=0.85,
    errorbar=None
)


# g1.axes is a numpy array of Axes if there are multiple facets (e.g. col, row arguments used)
# Here, adapt to possibly multiple axes (e.g. for different modalities being faceted)

# This will flatten axes for 1D and 2D facetting (e.g. col or row, or both)
axes = g1.axes.flatten() if hasattr(g1, "axes") else [g1.ax]

for ax in axes:
    xticklabels = ax.get_xticklabels()
    new_labels = []
    for label in xticklabels:
        txt = label.get_text()
        new_labels.append(short_user_labels.get(txt, txt))
    ax.set_xticklabels(new_labels, fontsize=font_size)
    ax.set_xlabel("")

plt.ylim(0, 1)
plt.ylabel("human acceptance", fontsize=font_size)

g1._legend.remove()
axes[0].legend(loc='upper right', title='request difficulty', fontsize=font_size, title_fontsize=font_size)


# plt.savefig("../../../figures/model_performance_by_difficulty_{}.pdf".format(metric), bbox_inches='tight')

# Efficiency of edits, human vs model

In [ ]:
# Quality by time

# metric = "dino-v2_similarity"
metric = "rating_mean_human_instruction"

# Filter to rows with valid values for both axes
plot_data = df_edits[df_edits.gt_human != True].dropna(subset=['edit_duration', metric])

g1 = sns.relplot(
    plot_data,
    hue='model_users',
    hue_order=USER_ORDER,
    palette=USER_PALETTE,
    y=metric,
    x='edit_duration',
    kind='scatter',
    alpha=0.7,
    aspect=1.5,
    s=20,
    legend=False
)

# Add marker for mean position of each model user
ax = g1.ax if hasattr(g1, "ax") else g1.axes[0,0]
means = (
    plot_data
    .groupby('model_users')
    .agg(mean_x=('edit_duration', 'mean'),
         mean_y=(metric, 'mean'))
    .reset_index()
)
for idx, row in means.iterrows():
    if row['model_users'] not in USER_PALETTE:
        continue
    ax.scatter(
        row['mean_x'], row['mean_y'],
        label=row['model_users'],
        marker='X', s=150,
        color=USER_PALETTE[row['model_users']],
        zorder=10, alpha=0.8, edgecolor='black'
    )

ax.set_ylabel("instruction following rating (mean)")
ax.set_xlabel("duration of edit (seconds)")


# plt.savefig("../../../figures/model_performance_by_duration_{}.pdf".format(metric), bbox_inches='tight')

In [ ]:
# Quality by time

# metric = "dino-v2_similarity"
metric = "rating_mean_human_instruction"
# metric = "rating_mean_human_quality"


# Filter to rows with valid values for both axes
plot_data = df_edits[df_edits.gt_human != True].dropna(subset=['cost', metric])

g1 = sns.relplot(
    plot_data,
    hue='model_users',
    hue_order=USER_ORDER,
    palette=USER_PALETTE,
    y=metric,
    x='cost',
    kind='scatter',
    alpha=0.7,
    aspect=1.5,
    s=20,
    legend=False
)

# Add marker for mean position of each model user
ax = g1.ax if hasattr(g1, "ax") else g1.axes[0,0]
means = (
    plot_data
    .groupby('model_users')
    .agg(mean_x=('cost', 'mean'),
         mean_y=(metric, 'mean'))
    .reset_index()
)
for idx, row in means.iterrows():
    if row['model_users'] not in USER_PALETTE:
        continue
    ax.scatter(
        row['mean_x'], row['mean_y'],
        label=row['model_users'],
        marker='X', s=150,
        color=USER_PALETTE[row['model_users']],
        zorder=10, alpha=0.8, edgecolor='black'
    )

ax.set_ylabel("instruction following rating (mean)")
ax.set_xlabel("cost (USD)")

# plt.savefig("../../../figures/model_performance_by_cost_{}.pdf".format(metric), bbox_inches='tight')

In [ ]:
# show distribution of edit durations by editor

# Set font size variable
font_size = 20

# Updated model titles
model_titles = {
    "other_human": "Human Designer",
    "claude-sonnet-4.5_cadquery-script": "Claude Sonnet 4.5",
    "gemini-3-pro_cadquery-script": "Gemini 3 Pro",
    "gpt-5.2_cadquery-script": "GPT 5.2",
}

g1 = sns.displot(
    df_edits[df_edits.gt_human != True],
    hue='request_difficulty',
    hue_order=DIFFICULTY_ORDER,
    palette=DIFFICULTY_PALETTE,
    col='model_users',
    col_order=USER_ORDER[1:],
    x='edit_duration',
    kind='kde',
    fill=True,
    bw_adjust=1,
    clip=(0, None),
    common_norm=False,
    legend=False,
)

plt.xlim(0, 2100)

# Set the number of x-axis ticks to fewer ticks (e.g., 5) for each facet/subplot
for ax in g1.axes.flat:
    ax.xaxis.set_major_locator(plt.MaxNLocator(5))

# Remove the individual x-labels from each facet/subplot and update titles
for ax, col_name in zip(g1.axes.flat, USER_ORDER[1:]):
    ax.set_xlabel('')
    nice_title = model_titles.get(col_name, col_name)
    ax.set_title(nice_title, fontsize=font_size)

# Add a common x-label for the entire figure
g1.fig.text(0.5, 0.04, 'edit duration (seconds)', ha='center', va='center', fontsize=font_size)
for ax in g1.axes.flat:
    ax.set_ylabel(ax.get_ylabel().lower(), fontsize=font_size)
    ax.tick_params(axis='both', which='major', labelsize=font_size)

plt.savefig("../../../figures/distribution_of_edit_duration_by_difficulty_by_model.pdf", bbox_inches='tight')

## Metric comparison and human evaluations

In [ ]:
df_ratings['rater_type_metric'] = df_ratings['rater_type'].astype(str) + '_' + df_ratings['metric'].astype(str)

### Comparing metrics

In [ ]:
# Metric correlation matrix across all rating metrics
# Average scores per (edit_id, metric) to handle multiple raters per edit
ratings_pivot = (
    df_ratings
    .groupby(['edit_id', 'rater_type_metric'])['score']
    .mean()
    .reset_index()
    .pivot(index='edit_id', columns='rater_type_metric', values='score')
    .dropna()
)

metric_order = ['human_instruction', 'human_quality', 'vlm_instruction','vlm_quality', 'geometric_dino-v2_similarity', 'geometric_iou', 'geometric_chamfer_similarity']
ratings_pivot = ratings_pivot[metric_order]

corr = ratings_pivot.corr(method='spearman')

fig, ax = plt.subplots(figsize=(8, 6))
# Create heatmap and retrieve colorbar
heatmap = sns.heatmap(
    corr,
    annot=True,
    fmt='.2f',
    cmap='YlGnBu',
    vmin=0, vmax=1,
    square=True,
    linewidths=.8,
    ax=ax,
    annot_kws={"fontsize": 20},  # Increase annotation font size
    # cbar_kws={'shrink': 0.7}  # Increase colorbar size by reducing the shrink factor (closer to 1 means bigger)
)
# Alternatively, if you want the colorbar to be even larger, set shrink closer to 1.0:
# cbar_kws={'shrink': 0.9}


# Increase tick label font sizes
ax.tick_params(axis='x', labelsize=14)
ax.tick_params(axis='y', labelsize=14)

# remove x and y ticks and labels
ax.set_xticks([])
ax.set_yticks([])
ax.set_xlabel('')
ax.set_ylabel('')

# ax.set_title('Correlation matrix of rating metrics (averaged per edit)')
plt.tight_layout()
# plt.savefig("../../../figures/metric_correlation_matrix.pdf", bbox_inches='tight')

### Measuring agreement of human raters

In [ ]:
# setup

import pingouin as pg

instruction_threshold = (5-1) / 6.0
quality_threshold = (5-1) / 6.0

df_human_ratings = df_ratings[df_ratings.rater_type == 'human'].copy()


# Create a new df where each row is an (edit_id, rater_id) pair,
# with columns for their 'instruction' and 'quality' scores,
# and a new 'acceptance' metric as described.
# Assume 'metric' column contains "instruction" and "quality"
acceptance_rows = (
    df_human_ratings[df_human_ratings['metric'].isin(['instruction', 'quality'])]
    .pivot_table(index=['edit_id', 'rater_id'], columns='metric', values='score')
    .reset_index()
    .dropna(subset=['instruction', 'quality'])
)
acceptance_rows['metric'] = 'acceptance'
acceptance_rows['score'] = (
    (acceptance_rows['instruction'] >= instruction_threshold) &
    (acceptance_rows['quality'] >= quality_threshold)
).astype(int)
# Only keep the desired columns
acceptance_df = acceptance_rows[['edit_id', 'rater_id', 'metric', 'score']]
acceptance_df

In [ ]:
# Inter rater agreement for instruction following and quality


# Metric correlation matrix across all rating metrics
# Average scores per (edit_id, metric) to handle multiple raters per edit
metric = 'instruction' # or swap for 'quality'

ratings_pivot = (
    df_ratings[(df_ratings.rater_type == 'human') & (df_ratings.metric == metric)]
    .groupby(['edit_id', 'rater_id'])['score']
    .mean()
    .reset_index()
    .pivot(index='edit_id', columns='rater_id', values='score')
    .dropna()
)

# metric_order = ['human_instruction', 'human_quality','vlm_instruction','vlm_quality', 'geometric_dino-v2_similarity', 'geometric_iou', 'geometric_chamfer_similarity']
# ratings_pivot = ratings_pivot[metric]

corr = ratings_pivot.corr(method='spearman')

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    corr,
    annot=True,
    fmt='.2f',
    cmap='hot',
    vmin=0, vmax=1,
    square=True,
    linewidths=0.5,
    ax=ax
)

# ax.set_title('Correlation matrix of rating metrics (averaged per edit)')
plt.tight_layout()
# plt.savefig("../../../figures/metric_correlation_matrix.pdf", bbox_inches='tight')



# Convert wide → long format for pingouin
icc_data = (
    ratings_pivot
    .reset_index()
    .melt(id_vars='edit_id', 
          var_name='rater', 
          value_name='score')
)

icc = pg.intraclass_corr(
    data=icc_data,
    targets='edit_id',   # items
    raters='rater',      # raters
    ratings='score'
)

print(icc)


In [ ]:
# inter rater agreement for acceptance

metric = 'acceptance'

ratings_pivot = (
    acceptance_df
    .groupby(['edit_id', 'rater_id'])['score']
    .mean()
    .reset_index()
    .pivot(index='edit_id', columns='rater_id', values='score')
    .dropna()
)

# metric_order = ['human_instruction', 'human_quality','vlm_instruction','vlm_quality', 'geometric_dino-v2_similarity', 'geometric_iou', 'geometric_chamfer_similarity']
# ratings_pivot = ratings_pivot[metric]

corr = ratings_pivot.corr(method='spearman')

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    corr,
    annot=True,
    fmt='.2f',
    cmap='hot',
    vmin=0, vmax=1,
    square=True,
    linewidths=0.5,
    ax=ax
)

# ax.set_title('Correlation matrix of rating metrics (averaged per edit)')
plt.tight_layout()
# plt.savefig("../../../figures/metric_correlation_matrix.pdf", bbox_inches='tight')



# Convert wide → long format for pingouin
icc_data = (
    ratings_pivot
    .reset_index()
    .melt(id_vars='edit_id', 
          var_name='rater', 
          value_name='score')
)

icc = pg.intraclass_corr(
    data=icc_data,
    targets='edit_id',   # items
    raters='rater',      # raters
    ratings='score'
)

print(icc)